# Assignment

2. Add comments in your own words and explain design choices such as
    - number of [layers](#01), 
    - [width](#02) of layers, 
    - number of [epochs](#03), 
    - [activation functions](#04), 
    - [loss function](#05), 
    - [gradient descent function](#06), 
    - [regularization function](#07)
3. Run the [code](#1). Evaluate the performance by discussing the results of the evaluation metrics. What hyper parameters would you recommend to change? Explain your choices. 
4. How do I set up a `batch_size` and how does it effect the outcome? Why do you think the batch_size was not set in the first place?
5. (Optional) Would there be a possibility to execute cross validation? How? 
6. (Optional) How can I introduce a validation test set? What would I need to change in the code?
7. Study the [tensor](#2) text. Consider a dataset of breast cancer images. What needs to be changed to the deep learning model design to make a model based on pictures? You can answer this in words, but if you like you can also try to code the solution. 

-----

### Layers and Their Dimensions:
This model comprises 3 total layers. The initial two are hidden layers configured with 20 and 10 nodes respectively, while the concluding layer is the output layer, embodying a single node.

The inclusion of multiple layers helps the model in deciphering more complex data patterns.

The technique of incrementally decreasing the count of nodes in succeeding layers is a standard practice in deep learning as it compels the model to grasp a simplified version of the input data.

### Epoch Count:
The training process for the model spans over 100 epochs.
The selection of the number of epochs is usually dependent on the complexity of the task, the data quantity available, and the data scientist's proficiency and experience.

### Activation Functions:
The ReLU (Rectified Linear Unit) activation function is applied in the hidden layers. The output layer, however, uses the sigmoid activation function, a common choice for binary classification tasks as it provides probabilities within the range of 0 and 1.

### Loss function:
The model adopts binary cross entropy as its loss function, a frequently chosen option for binary classification tasks.

### Optimization Technique:
The model employs the Adam optimizer for gradient descent. This combines the benefits of two other SGD (stochastic gradient descent) variations: AdaGrad and RMSProp.
Adam customizes the learning rate for each weight in the model individually, enabling adaptive learning rates for different parameters.

### Regularization Technique:
The model uses Dropout as a regularization method. It randomly nullifies a portion of input units at each update during training, aiding in the prevention of overfitting

<a name='1'></a>
# Study Case

Consider the Breast Cancer Wisconsin (Diagnostic) Data Set. UCI Machine Learning Repository: Breast Cancer Wisconsin (diagnostic) data set. (n.d.). https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+%28Diagnostic%29. Consider the code below. 


In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# Load the dataset
df = pd.read_csv('breast-cancer.csv')

# # Preprocess the labels: Convert categorical variable into dummy/indicator variables
le = LabelEncoder()
le.fit(df['diagnosis'])
df['diagnosis'] = le.transform(df['diagnosis'])

# Split the data into features and labels
X = df[df.columns[2:-1]]
y = df['diagnosis']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.20, random_state=42)

# Normalize the features using MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Define the model architecture
model = Sequential()

# Add first hidden layer with 20 neurons
# Add 'relu' activation function
model.add(Dense(20, activation='relu'))
# Add dropout layer to prevent overfitting
model.add(Dropout(0.5))

# Add second hidden layer with 10 neurons and 'relu' activation function
model.add(Dense(10, activation='relu'))
# Add dropout layer to prevent overfitting
model.add(Dropout(0.5))

# Add output layer with 'sigmoid' activation function for binary classification
model.add(Dense(1, activation='sigmoid'))

# Compile the model with binary cross entropy loss function and Adam optimizer
model.compile(loss='binary_crossentropy', optimizer='adam')

# Train the model for 100 epochs
model.fit(x=X_train, y=y_train, epochs=100, validation_data=(X_test, y_test))

# Plot the loss during training
model_loss = pd.DataFrame(model.history.history)
model_loss.plot()


# Make predictions and print the classification report
predicted=(model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, predicted))

# Plot the confusion matrix
confusion_matrix = confusion_matrix(y_test, predicted)
cm_display = ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, 
                                    display_labels = [False, True])

cm_display.plot()
plt.show()

-----

# Performance evaluation and hyperparameter tuning:

The model exhibits solid performance, with a low final validation loss of 0.0624 and a high test set accuracy of 0.98. This implies that it accurately categorizes 98% of the test set samples. Both classes have high precision, recall, and F1-score (0.98), indicating a balanced model that doesn't favor a single class.

To further experiment:

Layers and nodes: Varying these might aid in identifying more intricate data patterns or avert potential overfitting.

Batch size: Tweaking the batch size can affect the model's generalization or speed up training.

Epochs: If the model isn't converging or is overfitting, modifying the epoch number or applying early stopping could help.

Dropout rate: For overfitting or underfitting, adjusting the dropout rate can introduce more or less regularization.

Optimizer: While Adam is a great pick, alternatives like SGD (with or without momentum) or RMSProp could be considered.

--------

# Batch size setup and effect:

Batch size is the number of samples processed before the model is updated. The size of a batch must be more than or equal to one and less than or equal to the number of samples in the training dataset.
When Batch size is not explicitly set, Keras will use the default batch size of 32. This number is a common choice that tends to work well in many scenarios
So we can change the batch_size:



----------

# Cross-validation:


With the KerasClassifier or KerasRegressor wrapper in scikit-learn, we can carry out K-fold cross-validation using the cross_val_score function.

We can include a validation set during the model.fit operation. If a separate validation set is needed, the training data should be split further.

For image data, convolutional layers (Conv2D for 2D) would be ideal. The input data needs to be reshaped for convolutional layers - use (batch_size, height, width, channels) for color and (batch_size, height, width, 1) for grayscale. MaxPooling2D layers could be added post-convolution for downsampling, along with Dropout or Batch Normalization to avoid overfitting. Before the dense output layer, the output needs to be flattened.

If the problem isn't binary classification, adjust the loss function and output layer's activation function. For multiclass, use 'categorical_crossentropy' as loss and 'softmax' as activation. For regression, use 'mse' or 'mae' as loss and no activation.

In [ ]:
from scikeras.wrappers import KerasRegressor, KerasClassifier
from sklearn.model_selection import cross_val_score
from keras.models import Sequential
from keras.layers import Dense, Dropout



# Define a function to create the model, required for KerasClassifier
def create_model():
    model = Sequential()
    model.add(Dense(20, input_dim=X_scaled.shape[1], activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(10, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam')
    return model

# create model
model = KerasClassifier(build_fn=create_model, epochs=50, verbose=0)

# Normalize the features using MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

# evaluate using 10-fold cross validation
results = cross_val_score(model, X_scaled, y, cv=10)
print(results.mean())



**The mean score of results: 0.9579 returned from the 5-fold cross-validation and it's accuracy of our model.
It means that on average, the model correctly classified about 95.79% of the instances across the five different splits of the data into training and testing sets.**

--------

# Validation test

For validation test we can split the data into three sets: training set, validation set, and test set.
The training set is used to train the model, the validation set is used to tune parameters and to decide when to stop training (early stopping), and the test set is used to evaluate the final model. 

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

# Split the data into training+validation set and test set
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Then split training+validation set into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42) 

# Scale the data (important for neural networks)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Create model
model = Sequential()
model.add(Dense(32, input_dim=X.shape[1], activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])

# Define the callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10)
model_checkpoint = ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)

# Fit the model
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, 
                    batch_size=10, callbacks=[early_stopping, model_checkpoint])

# Plot training & validation loss values
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper right')

# Plot training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Load the best saved model
model.load_weights('best_model.h5')

# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test)

print(f'Test accuracy: {test_acc}')


It can be observed that the model was trained for 24 epochs, but was designed to run up to 100 epochs. The training was likely stopped early due to an Early Stopping mechanism, which stops training when a monitored metric has stopped improving.

Key performance indicators are the 'loss', 'accuracy', 'val_loss', and 'val_accuracy' values.

'loss' and 'accuracy' indicate the performance of the model on the training data.
'val_loss' and 'val_accuracy' indicate the performance on the validation data, which is a subset of the training data not used in the actual training, serving as a proxy for test data.
The training accuracy increased from 0.6833 to 0.9824, and the validation accuracy increased from 0.9211 to 0.9649 over the 24 epochs, indicating that the model was learning from the data and improving its predictions.

The model's performance was then evaluated on a separate test dataset, achieving a test accuracy of approximately 97.37%. This suggests that the model generalized well from the training data and was able to make accurate predictions on unseen data.

-------------

# Model based on pictures:

 If the input data are images instead of structured data, the structure of the deep learning model needs to be changed. Instead of Dense layers, we usually use convolutional layers (Conv2D) for image data. Also, we need to ensure that the input shape matches the shape of the images.
 To adapt the code to work with image data, we would change the data preprocessing steps to handle images.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import Adam

# Set up the ImageDataGenerators like this:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True,
                                   validation_split=0.2) # set validation split

train_set = train_datagen.flow_from_directory('Dataset_BUSI_with_GT',
                                              target_size=(64, 64),
                                              batch_size=32,
                                              class_mode='categorical',
                                              subset='training') # set as training data

validation_set = train_datagen.flow_from_directory('Dataset_BUSI_with_GT', # same directory as training data
                                              target_size=(64, 64),
                                              batch_size=32,
                                              class_mode='categorical',
                                              subset='validation') # set as validation data

# Set up a simple CNN architecture like this:
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(3, activation='softmax')) # 3 output neurons for 3 classes

# Compile the model
opt = Adam(learning_rate=0.0001)
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(train_set,
                    epochs=100,
                    validation_data=validation_set)


This assignment has been done with help of Samaneh Shahpouri